In [2]:
import pandas as pd
import spacy
import numpy as np


model = spacy.load('en_core_web_trf', exclude=['ner', 'parser', 'transformer'])
model.pipeline

[('tagger', <spacy.pipeline.tagger.Tagger at 0x1640c3340>),
 ('attribute_ruler',
  <spacy.pipeline.attributeruler.AttributeRuler at 0x163dcf100>),
 ('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer at 0x1644922c0>)]

hello world this is a cell

In [ ]:
# load data
df = pd.read_sql(f'''
    select * from details
''', con='sqlite:///seek/jobs.db')

In [ ]:
pd.read_sql('select max(id) from words', con='sqlite:///seek/jobs.db')

In [ ]:
df[df['id'] == 58355866]

In [ ]:
def words_to_db(df):
    counter = 0
    total = len(df)
    for id, details in df.to_numpy():
        try:
            details = details.decode('utf-8')
        except:
            pass

        doc = model(details.lower())
        words = {}
        for token in doc:
            if not token.is_alpha:
                continue

            try:
                words[token.lemma_] += 1
            except:
                words[token.lemma_] = 1

        words_df = pd.DataFrame([(w, words[w]) for w in words], columns=['word', 'count'])
        words_df['id'] = id
        words_df.to_sql('words', con='sqlite:///seek/jobs.db', if_exists='append', index=False)
        counter += 1
        print('progress:', round(counter / total * 100, 2), '%  ', end='\r')
        
words_to_db(df.iloc[633086:])

In [ ]:
def inverse_document_frequency(bag):
    return np.log(len(df) / bag.groupby('word').nunique())


def document_frequency(small_bag):
    return small_bag.groupby('word').sum()


In [2]:
bag = pd.read_sql('''
    select words.id, words.word, words.count, jobs.sector from words
    join jobs on words.id = jobs.id
''', con='sqlite:///seek/jobs.db')

: 

: 

In [ ]:

idf = inverse_document_frequency(bag[['id', 'word']])['id']
freq = document_frequency(bag[['word', 'count']].iloc[: 200])['count']
(freq * idf).dropna().sort_values()[-20:]

In [12]:
pd.DataFrame({'word': [], 'id': [], 'count': []}).astype({'word': str})

,word,id,count
